In [3]:
!pip install pypdf langchain langchain-community sentence-transformers faiss-cpu --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 1.8 MB/s eta 0:00:00 MB/s eta 0:00:01
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 701.4 kB/s eta 0:00:000:00:0136m0:00:01
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 938.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 1.9 MB/s eta 0:00:003.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2

In [1]:
from pypdf import PdfReader
print("Pypdf imported successfully!")

Pypdf imported successfully!


In [2]:
from pypdf import PdfReader
pdf_path = "sample_data/RAG_Single_Page_Document.pdf"
reader = PdfReader(pdf_path)
text = ""
for page in reader.pages:
    text += page.extract_text()
print(text)

Artificial Intelligence and Machine Learning
Artificial Intelligence (AI) is a field of computer science focused on creating systems that can
perform tasks requiring human intelligence, such as learning, reasoning, decision-making, and
language understanding. AI is widely used in virtual assistants, recommendation systems,
healthcare, autonomous vehicles, and fraud detection.
Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data and
make predictions without being explicitly programmed. The three major types of machine learning
are supervised learning, unsupervised learning, and reinforcement learning. Common algorithms
include Linear Regression, Decision Trees, Random Forests, Support Vector Machines, and Neural
Networks.
AI and ML provide benefits such as automation, increased efficiency, improved decision-making,
and the ability to analyze large amounts of data. However, challenges include data privacy
concerns, algorithmic bias, lack of transpare

In [6]:
!pip show langchain
!pip show langchain-community

Name: langchain
Version: 1.3.2
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /home/nineleaps/.local/lib/python3.12/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /home/nineleaps/.local/lib/python3.12/site-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 


In [8]:
chunk_size = 300
overlap = 50
chunks = []
for i in range(0, len(text), chunk_size - overlap):
    chunk = text[i:i + chunk_size]
    chunks.append(chunk)
print("Number of chunks:", len(chunks))

Number of chunks: 5


In [9]:
for i, chunk in enumerate(chunks):
    print(f"\n----- Chunk {i+1} -----\n")
    print(chunk)


----- Chunk 1 -----

Artificial Intelligence and Machine Learning
Artificial Intelligence (AI) is a field of computer science focused on creating systems that can
perform tasks requiring human intelligence, such as learning, reasoning, decision-making, and
language understanding. AI is widely used in virtual assistants,

----- Chunk 2 -----

standing. AI is widely used in virtual assistants, recommendation systems,
healthcare, autonomous vehicles, and fraud detection.
Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data and
make predictions without being explicitly programmed. The three major types 

----- Chunk 3 -----

eing explicitly programmed. The three major types of machine learning
are supervised learning, unsupervised learning, and reinforcement learning. Common algorithms
include Linear Regression, Decision Trees, Random Forests, Support Vector Machines, and Neural
Networks.
AI and ML provide benefits such

----- Chunk 4 -----

d Neural


In [10]:
from sentence_transformers import SentenceTransformer

/home/nineleaps/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 2777.40it/s]


In [12]:
embeddings = embedding_model.encode(chunks)
print("Embedding Shape:", embeddings.shape)

Embedding Shape: (5, 384)


In [13]:
import faiss
import numpy as np

In [15]:
embeddings = np.array(embeddings).astype("float32")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Number of vectors in FAISS:", index.ntotal)

Number of vectors in FAISS: 5


In [16]:
query = "What is Machine Learning?"

In [17]:
query_embedding = embedding_model.encode([query])

query_embedding = np.array(query_embedding).astype("float32")

k = 2

distances, indices = index.search(query_embedding, k)

print(indices)

[[1 2]]


In [18]:
for idx in indices[0]:
    print("\nRetrieved Chunk:\n")
    print(chunks[idx])


Retrieved Chunk:

standing. AI is widely used in virtual assistants, recommendation systems,
healthcare, autonomous vehicles, and fraud detection.
Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data and
make predictions without being explicitly programmed. The three major types 

Retrieved Chunk:

eing explicitly programmed. The three major types of machine learning
are supervised learning, unsupervised learning, and reinforcement learning. Common algorithms
include Linear Regression, Decision Trees, Random Forests, Support Vector Machines, and Neural
Networks.
AI and ML provide benefits such


In [19]:
retrieved_context = ""

for idx in indices[0]:
    retrieved_context += chunks[idx] + "\n"

In [20]:
print("Question:", query)

print("\nAnswer:\n")

print(retrieved_context)

Question: What is Machine Learning?

Answer:

standing. AI is widely used in virtual assistants, recommendation systems,
healthcare, autonomous vehicles, and fraud detection.
Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data and
make predictions without being explicitly programmed. The three major types 
eing explicitly programmed. The three major types of machine learning
are supervised learning, unsupervised learning, and reinforcement learning. Common algorithms
include Linear Regression, Decision Trees, Random Forests, Support Vector Machines, and Neural
Networks.
AI and ML provide benefits such



In [ ]:
while True:
    query = input("Ask a question (or type 'exit'): ")

    if query.lower() == "exit":
        break

    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k=2)

    retrieved_context = ""

    for idx in indices[0]:
        retrieved_context += chunks[idx] + "\n"

    print("\nAnswer:\n")
    print(retrieved_context)
    print("\n" + "-"*50 + "\n")

Ask a question (or type 'exit'):  What is AI?



Answer:

Artificial Intelligence and Machine Learning
Artificial Intelligence (AI) is a field of computer science focused on creating systems that can
perform tasks requiring human intelligence, such as learning, reasoning, decision-making, and
language understanding. AI is widely used in virtual assistants,
standing. AI is widely used in virtual assistants, recommendation systems,
healthcare, autonomous vehicles, and fraud detection.
Machine Learning (ML) is a subset of AI that enables computers to learn patterns from data and
make predictions without being explicitly programmed. The three major types 


--------------------------------------------------

